## Completing the Tool Use Cycle

# Unit 23: Complete Orchestration Loops, Execution Handlers, and State Resubmission

## Introduction & Overview

In the previous modules, you learned how to design structural tool schemas and parse Claude's responsive payloads when it decides to employ your functions. You can now programmatically recognize when Claude requests tool execution via the `stop_reason: "tool_use"` directive and cleanly unpack specific parameters from the content array blocks.

However, detecting what Claude wants to execute is only half the battle—you still need to execute those backend functions locally, format their string outputs, and resubmit them to close the agent conversation cycle.

In this lesson, you will learn how to build the execution loop that triggers local Python code requested by Claude, handles runtime errors gracefully, maps parameters dynamically, and builds the conversation state required for Claude to formulate its final conversational answer.

---

## The Complete Tool Execution Lifecycle

Before examining the code layers, let's trace the full multi-turn transaction cycle. This structure coordinates your local Python workspace with the Anthropic inference loop:

### Step 1: Foundation Provisioning

Initialize your connection clients, read structural JSON manifests from disk, and build your internal inversion lookup map matching text string identifiers to local executable pointers.

### Step 2: The Initial Dispatched Request

Submit the user's raw prompt alongside your full array of tool schemas. This marks **Turn 1 (User)** of the conversation thread.

### Step 3: Tool Call Detection & State Preservation

Claude evaluates the intent and answers with an array containing conversational thoughts and explicit `tool_use` instruction blocks. This forms **Turn 2 (Assistant)**. Your local application code intercepts this payload and appends it immediately to your conversation array to preserve historical context.

### Step 4: Extraction & Dynamic Local Invocation

The loop scans Claude's response for `stop_reason == "tool_use"`. It extracts target function names, arguments, and tracking IDs, then runs the local Python logic using keyword argument expansion (`kwargs`).

### Step 5: Result Formatting & State Resubmission

The values returned by your local code are converted to strings, wrapped inside a `tool_result` payload matching the tool's execution ID, and bundled into a single message array. This forms **Turn 3 (User)**. This updated dataset is passed back to Claude in a second API call.

### Step 6: Final Context-Aware Synthesis

Claude evaluates the raw function execution values to write a natural language summary. This concludes the process at **Turn 4 (Assistant)**.

---

## Implementing the End-to-End Execution Loop

Let's look at the complete runtime implementation in a script called `agent.py`. This script handles tool initialization, dynamic lookup routing, defensive error handling, and final context synthesis.

```python
import json
from anthropic import Anthropic
from functions import sum_numbers, multiply_numbers

# =====================================================================
# Step 1: Foundation Provisioning & Inversion Map SETUP
# =====================================================================
client = Anthropic()
model = "claude-sonnet-4-6"

# Internal execution map linking schema string identifiers to code pointers
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers
}

system_prompt = (
    "You are a helpful math assistant. "
    "Always use the available tools to perform calculations accurately."
)

# Load schemas from disk
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Initialize conversation tracking array (Turn 1 - User)
messages = [
    {"role": "user", "content": "Please calculate 15 + 27"}
]

# =====================================================================
# Step 2: Dispatch Initial Inference Turn (Turn 1 -> Turn 2)
# =====================================================================
response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    system=system_prompt,
    tools=tool_schemas
)

# Immediately preserve Claude's response payload (Turn 2 - Assistant)
messages.append({
    "role": "assistant",
    "content": response.content
})

# =====================================================================
# Step 3: Core Tool Execution Ring & Error Handlers
# =====================================================================
if response.stop_reason == "tool_use":
    # Array to collect all individual function outputs generated in this turn
    tool_results = []
    
    # Iterate through the content array to find requested operations
    for content_item in response.content:
        if content_item.type == "tool_use":
            tool_name = content_item.name
            tool_input = content_item.input
            tool_id = content_item.id
            
            print(f"⚙️ Local Runtime Intercept -> Executing: {tool_name}({tool_input})")
            
            # Defensive execution block to handle missing keys or runtime bugs
            try:
                if tool_name in tools:
                    # Dynamically unpack input arguments dictionary into the local function
                    result = tools[tool_name](**tool_input)
                else:
                    result = f"Error: The function '{tool_name}' is missing from the local lookup map."
            except Exception as e:
                result = f"Runtime Exception executing '{tool_name}': {str(e)}"
                
            print(f"✨ Execution Output -> Resolved Value: {result}")
            
            # Append a structured tool_result block linking back to the tool_use ID
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tool_id,
                "content": str(result)  # Ensure content is strictly stringified
            })
            
    # Step 4: Append the collected results as a single user turn (Turn 3 - User)
    messages.append({
        "role": "user",
        "content": tool_results
    })
    
    # =====================================================================
    # Step 5: Resubmit Consolidated State for Synthesis (Turn 3 -> Turn 4)
    # =====================================================================
    final_response = client.messages.create(
        model=model,
        max_tokens=2000,
        messages=messages,
        system=system_prompt,
        tools=tool_schemas
    )
    
    # Preserve final conversational statement (Turn 4 - Assistant)
    messages.append({
        "role": "assistant",
        "content": final_response.content
    })
    
    print("\n🏁 Claude's Final Synthesized Response:")
    print(final_response.content[0].text)

# =====================================================================
# Step 6: Handling Non-Tool Conversational Fallbacks
# =====================================================================
elif response.stop_reason == "end_turn":
    print("\n🟢 Claude Handled Request Directly (No Tool Required):")
    print(response.content[0].text)

```

---

## Deconstructing Critical Execution Constraints

### 1. Dictionary Unpacking via Keyword Expansion (`kwargs`)

The expression `tools[tool_name](tool_input)` performs two important operations dynamically:

* It looks up the target function pointer using the string name returned by Claude (`tools["sum_numbers"]`).
* It uses the `` operator to unpack Claude's input parameter dictionary directly into keyword arguments (`a=15, b=27`). This keeps your tool infrastructure flexible, allowing the same loop to handle functions with completely different parameter structures without hardcoded modifications.

### 2. Guardrails and Exception Safety

A robust execution loop must run inside defensive `try-except` blocks. If a tool triggers a runtime exception (e.g., an unexpected database timeout or division by zero), the error is caught, converted into a string, and sent safely back to Claude. This tells the model exactly what went wrong, allowing it to explain the error to the user or attempt a recovery steps with an alternative tool call instead of crashing the program.

### 3. Strict Structural Symmetry

When Claude requests a tool call, your application **must return a corresponding result for every single requested ID**. Leaving an unfulfilled `tool_use_id` out of the subsequent resubmission payload violates the Anthropic API's state contract and will immediately cause the server to throw an error on your next message creation call. This is why successful executions and errors are handled identically; Claude must receive clear confirmation on the status of every tool it attempts to run.

---

## Analyzing Console and Conversational States

When running this complete program loop, your console tracking logs display the execution path clearly:

```text
⚙️ Local Runtime Intercept -> Executing: sum_numbers({'a': 15, 'b': 27})
✨ Execution Output -> Resolved Value: 42

🏁 Claude's Final Synthesized Response:
The result of adding 15 and 27 is 42.

```

### Deconstructing the Message History States

To see how context is preserved, look at the underlying layout of the updated `messages` list across the four-turn process:

```text
======================= CONVERSATION MATRIX TRACE =======================

Turn 1 [Role: user]
└── Content (String): "Please calculate 15 + 27"

Turn 2 [Role: assistant]
├── Content Block 1 (Text)    : "I will call the summation tool to calculate this accurately."
└── Content Block 2 (Tool Use): {"id": "toolu_01WoJ...", "name": "sum_numbers", "input": {"a": 15, "b": 27}}

Turn 3 [Role: user]
└── Content Block 1 (Tool Res): {"tool_use_id": "toolu_01WoJ...", "content": "42"}

Turn 4 [Role: assistant]
└── Content Block 1 (Text)    : "The result of adding 15 and 27 is 42."

```

Notice the format changes within the array: basic user prompts start as simple strings, while SDK responses and tool outputs are stored as lists of structured content blocks. This conversation history layout gives Claude the exact historical context it needs to trace its own logic, verify the returned answers, and write natural language summaries.

---

## Summary & Practice Preparation

You have constructed a complete, production-ready function execution pipeline for Claude Code agents. By matching incoming text tokens to an internal mapping table, capturing runtime outputs within exception guardrails, and resubmitting structured result contexts, you can transition your standalone utilities into an interactive, multi-turn AI ecosystem.

Let's head into the practical lab modules to stand up, secure, and test your own full-lifecycle autonomous agent loops!

## Detecting Claude Tool Use Requests

Now that you understand how Claude signals its intentions through tool schemas and responses, it's time to build a complete detection and parsing system. In this exercise, you'll implement the crucial first steps of the tool execution cycle by learning how to handle Claude's responses, detect tool use requests, and extract the specific tool information when needed.

Your task is to:

    Add the content of Claude's response to the messages array using the proper structure with the "assistant" role
    Create an if-else statement that checks the stop_reason field from Claude's response and:
        If the stop reason equals "tool_use", proceed to extract tool information
        If it's anything else, print "Claude doesn't want to use tools:" and display the response text
    Create a for loop that iterates through each item in response.content and:
        Check if each content item has the type "tool_use"
        For each tool use block you find, extract the tool name, input parameters, and unique id
        Print all three pieces of information in a clear, formatted message

This complete parsing skill prepares you for actually executing the tools that Claude requests in future exercises!

```
import json
from anthropic import Anthropic
from functions import sum_numbers, multiply_numbers

# Initialize the Anthropic client
client = Anthropic()

# Choose a Claude model
model = "claude-sonnet-4-6"

# System prompt with explicit instructions to use tools
system_prompt = (
    "You are a helpful math assistant. "
    "Always use the available tools to perform calculations accurately."
)

# Load the schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a dictionary mapping tool names to functions
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers
}

# Create a list of messages to send to Claude
messages = [
    {"role": "user", "content": "Please calculate 15 + 27"}
]

# Send the messages to Claude
response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    system=system_prompt,
    tools=tool_schemas
)

# TODO: Add Claude's response to the messages array
    # - Use the "assistant" role
    # - Set the content to response.content

# TODO: Check if Claude wants to use tools
    # TODO: Create a for loop to iterate through response.content
    
        # TODO: Check if content_item.type equals "tool_use"
            # TODO: Extract tool_name from content_item.name
            # TODO: Extract tool_input from content_item.input
            # TODO: Extract tool_id from content_item.id
            
            # TODO: Print all three pieces of information in a clear, formatted message
            
# TODO: Handle the case where Claude doesn't want to use tools
    # - Print "Claude doesn't want to use tools:" and the response text
```

To programmatically capture and log Claude's tool requests, you need to manage two key areas: **state preservation** and **conditional flow control**.

By appending Claude's response to your history list, you establish the mandatory historical footprint required for multi-turn agent execution. Checking for `response.stop_reason == "tool_use"` then acts as your control switch, signaling whether you need to parse structured argument payloads or fallback to simple text rendering.

Here is the complete script with your response preservation, loop detection, and block extraction logic fully implemented:

```python
import json
from anthropic import Anthropic
from functions import sum_numbers, multiply_numbers

# Initialize the Anthropic client
client = Anthropic()

# Choose a Claude model
model = "claude-sonnet-4-6"

# System prompt with explicit instructions to use tools
system_prompt = (
    "You are a helpful math assistant. "
    "Always use the available tools to perform calculations accurately."
)

# Load the schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a dictionary mapping tool names to functions
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers
}

# Create a list of messages to send to Claude
messages = [
    {"role": "user", "content": "Please calculate 15 + 27"}
]

# Send the messages to Claude
response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    system=system_prompt,
    tools=tool_schemas
)

# Add Claude's response to the messages array to maintain conversation history
messages.append({
    "role": "assistant",
    "content": response.content
})

# Check if Claude wants to use tools by evaluating the stop reason
if response.stop_reason == "tool_use":
    print("🎯 Tool Use Detected! Intercepting request for local execution...\n")
    
    # Create a for loop to iterate through response.content with enumeration
    for i, content_item in enumerate(response.content, start=1):
        
        # Check if content_item.type equals "tool_use"
        if content_item.type == "tool_use":
            # Extract the three critical tracking parameters from the block object
            tool_name = content_item.name
            tool_input = content_item.input
            tool_id = content_item.id
            
            # Print all three pieces of information in a clear, formatted message
            print(f"[Tool Request Block {i}]")
            print(f"🛠️ Tool Target : {tool_name}")
            print(f"📥 Arguments   : {tool_input}")
            print(f"🆔 Tracking ID : {tool_id}\n")
            
else:
    # Handle the case where Claude doesn't want to use tools
    print("Claude doesn't want to use tools:")
    print(response.content[0].text)

```

### Critical Execution Insights:

* **Immediate State Preservation:** Appending `response.content` under the `"assistant"` role right after the API call is complete is an absolute requirement. Even though the turn is truncated due to a `tool_use` event, Claude needs to see its own generated thoughts and tool calls in the message log when you resubmit the data later.
* **The Structural Mapping Variable:** Isolating `content_item.name` provides the string token needed to look up function pointers in your local `tools` lookup map, while matching against `content_item.input` isolates the clean dictionary payload ready for keyword unpacking (`kwargs`).

## Executing Tools and Formatting Results

Excellent work extracting tool information from Claude's responses! Now comes the exciting part, where we bridge the gap between Claude's requests and actual function execution.

Your mission is to take those extracted tool details and use them to execute the real Python functions, then format the results so Claude can understand them. Here's what you need to implement:

    Add a try-except block to handle both successful tool execution and any errors that might occur.
    Check if the requested tool exists in your tools dictionary and execute it using tools[tool_name](**tool_input) to unpack the parameters.
    If the tool doesn't exist, set the result to an appropriate error message (e.g., "Error: Function {tool_name} not found").
    If an exception occurs during execution, set the result to a formatted error message (e.g., "Error executing {tool_name}: {str(e)}").
    Print the result so you can see what happened.
    Append each tool result as a properly structured dictionary to the tool_results list with type "tool_result", the matching tool_use_id, and the stringified result content.
    After the loop completes, append all tool results as a single user message to the messages array with role "user" and content as the tool_results list.

This step is crucial because it completes the execution side of the tool use cycle, properly collecting all tool results and formatting them so Claude can process multiple tool executions in a single response!

```
import json
from anthropic import Anthropic
from functions import sum_numbers, multiply_numbers

# Initialize the Anthropic client
client = Anthropic()

# Choose a Claude model
model = "claude-sonnet-4-6"

# System prompt with explicit instructions to use tools
system_prompt = (
    "You are a helpful math assistant. "
    "Always use the available tools to perform calculations accurately."
)

# Load the schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a dictionary mapping tool names to functions
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers
}

# Create a list of messages to send to Claude
messages = [
    {"role": "user", "content": "Please calculate 15 + 27 and also multiply 6 by 8"}
]

# Send the messages to Claude
response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    system=system_prompt,
    tools=tool_schemas
)

# Add the assistant's response to messages
messages.append({
    "role": "assistant", 
    "content": response.content
})

# Check if Claude wants to use tools
if response.stop_reason == "tool_use":
    # Create a list to collect all tool results
    tool_results = []
    
    # Iterate through each content block in Claude's response
    for content_item in response.content:
        # Check if this content item is a tool use request
        if content_item.type == "tool_use":
            # Extract the tool name, input, and ID
            tool_name = content_item.name
            tool_input = content_item.input
            tool_id = content_item.id
            
            # Print the extracted information
            print(f"Tool: {tool_name}, Input: {tool_input}, ID: {tool_id}")
            
            # TODO: Add a try-except block to handle tool execution and errors
            
                # TODO: Check if tool_name exists in the tools dictionary and execute it
                # Use tools[tool_name](**tool_input) to unpack the input parameters
                
                # TODO: Handle the case where the tool doesn't exist
                # Set result to an error message about the missing function
                
            # TODO: Handle execution exceptions in the except block
            # Format and set the result with the exception
            
            # TODO: Print the result for verification
            
            # TODO: Append a properly structured tool result dictionary to tool_results
            # - Dictionary should have type "tool_result", tool_use_id matching tool_id, and content as str(result)
            
    # TODO: After the loop, append all tool results as a single user message to messages
    # - Use role "user" and set content to the tool_results list
            
else:
    print("Claude doesn't want to use tools:")
    print(response.content[0].text)
```

To complete this part of the function calling pipeline, you need to implement **Dynamic Execution** and **Symmetric Response Formatting**.

Using the dictionary unpacking operator (`tool_input`) allows you to dynamically inject arguments into your local Python functions based on Claude's requests. Wrapping this inside a defensive `try-except` block ensures that missing tool registrations or native execution bugs (like syntax errors or invalid parameters) turn into string descriptions that Claude can analyze rather than causing the whole application to crash.

Here is the complete script with your local tool execution block and conversation payload construction fully implemented:

```python
import json
from anthropic import Anthropic
from functions import sum_numbers, multiply_numbers

# Initialize the Anthropic client
client = Anthropic()

# Choose a Claude model
model = "claude-sonnet-4-6"

# System prompt with explicit instructions to use tools
system_prompt = (
    "You are a helpful math assistant. "
    "Always use the available tools to perform calculations accurately."
)

# Load the schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a dictionary mapping tool names to functions
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers
}

# Create a list of messages to send to Claude
messages = [
    {"role": "user", "content": "Please calculate 15 + 27 and also multiply 6 by 8"}
]

# Send the messages to Claude
response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    system=system_prompt,
    tools=tool_schemas
)

# Add the assistant's response to messages to preserve conversation history
messages.append({
    "role": "assistant", 
    "content": response.content
})

# Check if Claude wants to use tools
if response.stop_reason == "tool_use":
    # Create a list to collect all tool results generated in this turn
    tool_results = []
    
    # Iterate through each content block in Claude's response
    for content_item in response.content:
        # Check if this content item is a tool use request
        if content_item.type == "tool_use":
            # Extract the tool name, input, and ID
            tool_name = content_item.name
            tool_input = content_item.input
            tool_id = content_item.id
            
            print(f"\n⚙️ Executing: {tool_name}({tool_input}) [ID: {tool_id}]")
            
            # Add a try-except block to handle tool execution and errors defensively
            try:
                # Check if tool_name exists in the tools dictionary and execute it
                if tool_name in tools:
                    # Unpack the input parameters dictionary as keyword arguments (**kwargs)
                    result = tools[tool_name](**tool_input)
                else:
                    # Handle the case where the tool doesn't exist in our dictionary
                    result = f"Error: Function '{tool_name}' not found in local tool registry."
            
            except Exception as e:
                # Handle execution exceptions in the except block
                result = f"Error executing '{tool_name}': {str(e)}"
            
            # Print the result for verification
            print(f"✨ Result: {result}")
            
            # Append a properly structured tool result dictionary to tool_results list
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tool_id,    # Explicitly links the output to Claude's request ID
                "content": str(result)     # Anthropic API requires results to be strictly stringified
            })
            
    # After the loop finishes, append all tool results as a single user turn to messages
    messages.append({
        "role": "user",
        "content": tool_results
    })
    
    print("\n✅ Success: All requested tools executed and formatted into conversation history.")
            
else:
    print("Claude doesn't want to use tools:")
    print(response.content[0].text)

```

### Critical Architecture Reminders:

* **Symmetric Tracking Requirement:** Notice that `tool_results` is configured as a list of independent dictionary envelopes inside a single appended message object. If Claude decides to run multiple tools in parallel (as it will for this multi-operation prompt), your system must send back an array matching **every requested tool ID** at once. Missing or skipping a required ID will trigger a runtime validation crash on your next API transaction turn.
* **Stringification (`str(result)`):** The Anthropic API specifies that the `"content"` property inside a `"type": "tool_result"` structure must be an explicit string value. Converting numbers (like `42` or `48`) or dictionary error logs using `str()` keeps the communication data layout completely stable.

## Completing Tool Execution Cycle

Perfect! You've successfully executed Claude's tool requests and formatted the results — now it's time to close the loop and see Claude transform those raw outputs into a natural, conversational response.

Your final mission is to complete the tool execution cycle by sending the updated conversation back to Claude and displaying the polished result. The messages array now contains everything Claude needs: the original user question, Claude's tool use request, and your tool execution results.

Here's what you need to implement:

    Make a second API call to Claude using the exact same parameters as your first call (model, max_tokens, messages, system, tools), but now with the updated messages array that includes the tool results.
    Add Claude's final response to the messages array with the "assistant" role to maintain proper conversation history.
    Extract and print Claude's final response text to show how it incorporates the tool results into a natural answer.

This final step demonstrates the true power of Claude agents — transforming raw function outputs into intelligent, contextual responses that directly answer user questions!

```
import json
from anthropic import Anthropic
from functions import sum_numbers, multiply_numbers

# Initialize the Anthropic client
client = Anthropic()

# Choose a Claude model
model = "claude-sonnet-4-6"

# System prompt with explicit instructions to use tools
system_prompt = (
    "You are a helpful math assistant. "
    "Always use the available tools to perform calculations accurately."
)

# Load the schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a dictionary mapping tool names to functions
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers
}

# Create a list of messages to send to Claude
messages = [
    {"role": "user", "content": "Please calculate 15 + 27"}
]

# Send the messages to Claude
response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    system=system_prompt,
    tools=tool_schemas
)

# Add the assistant's response to messages
messages.append({
    "role": "assistant", 
    "content": response.content
})

# Check if Claude wants to use any functions and execute them
if response.stop_reason == "tool_use":
    # Define a list to collect all tool results
    tool_results = []

    for content_item in response.content:
        # Check if the content item is a tool use
        if content_item.type == "tool_use":
            # Get the tool name, input, and id
            tool_name = content_item.name
            tool_input = content_item.input
            tool_id = content_item.id
            
            # Print the tool name, input
            print(f"Executing: {tool_name}({tool_input})")
            
            # Use try-except to handle both missing tools and execution errors
            try:
                # Check if the requested tool exists and execute it
                if tool_name in tools:
                    result = tools[tool_name](**tool_input)
                else:
                    result = f"Error: Function {tool_name} not found"
            except Exception as e:
                # Handle any errors that occur during function execution
                result = f"Error executing {tool_name}: {str(e)}"
     
            # Print the result
            print(f"Result: {result}")

            # Append a properly structured tool_result for this specific tool_use
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tool_id,
                "content": str(result)
            })

    # Add all tool results as a single user message
    messages.append({
        "role": "user",
        "content": tool_results
    })
    
    # TODO: Make a second API call to Claude with the updated messages array
    # Use the same parameters as the first call: model, max_tokens, messages, system, tools
    
    # TODO: Add Claude's final response to the messages array with role "assistant"
    
    # TODO: Print Claude's final response text using final_response.content[0].text
    
else:
    print("Claude doesn't want to use tools:")
    print(response.content[0].text)
```

Untuk menyelesaikan siklus eksekusi alat (*tool/function calling*) pada Anthropic API, Anda harus mengirimkan kembali seluruh riwayat percakapan yang kini telah diperbarui dengan hasil eksekusi fungsi (`tool_result`) menggunakan peran (`role`) sebagai `user`.

Berikut adalah implementasi kode lengkap untuk mengisi bagian `TODO` di dalam blok penanganan `if response.stop_reason == "tool_use":`:

```python
import json
from anthropic import Anthropic
from functions import sum_numbers, multiply_numbers

# Initialize the Anthropic client
client = Anthropic()

# Choose a Claude model
model = "claude-sonnet-4-6"

# System prompt with explicit instructions to use tools
system_prompt = (
    "You are a helpful math assistant. "
    "Always use the available tools to perform calculations accurately."
)

# Load the schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a dictionary mapping tool names to functions
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers
}

# Create a list of messages to send to Claude
messages = [
    {"role": "user", "content": "Please calculate 15 + 27"}
]

# Send the messages to Claude
response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    system=system_prompt,
    tools=tool_schemas
)

# Add the assistant's response to messages
messages.append({
    "role": "assistant", 
    "content": response.content
})

# Check if Claude wants to use any functions and execute them
if response.stop_reason == "tool_use":
    # Define a list to collect all tool results
    tool_results = []

    for content_item in response.content:
        # Check if the content item is a tool use
        if content_item.type == "tool_use":
            # Get the tool name, input, and id
            tool_name = content_item.name
            tool_input = content_item.input
            tool_id = content_item.id
            
            # Print the tool name, input
            print(f"Executing: {tool_name}({tool_input})")
            
            # Use try-except to handle both missing tools and execution errors
            try:
                # Check if the requested tool exists and execute it
                if tool_name in tools:
                    result = tools[tool_name](**tool_input)
                else:
                    result = f"Error: Function {tool_name} not found"
            except Exception as e:
                # Handle any errors that occur during function execution
                result = f"Error executing {tool_name}: {str(e)}"
     
            # Print the result
            print(f"Result: {result}")

            # Append a properly structured tool_result for this specific tool_use
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tool_id,
                "content": str(result)
            })

    # Add all tool results as a single user message
    messages.append({
        "role": "user",
        "content": tool_results
    })
    
    # 1. Melakukan panggilan API kedua menggunakan array messages yang telah diperbarui
    final_response = client.messages.create(
        model=model,
        max_tokens=2000,
        messages=messages,
        system=system_prompt,
        tools=tool_schemas
    )
    
    # 2. Menambahkan respons akhir Claude ke dalam histori percakapan (messages)
    messages.append({
        "role": "assistant",
        "content": final_response.content
    })
    
    # 3. Mencetak teks jawaban natural akhir dari Claude
    print("\nClaude's Final Answer:")
    print(final_response.content[0].text)
    
else:
    print("Claude doesn't want to use tools:")
    print(response.content[0].text)

```

### 🔁 Penjelasan Alur Siklus Akhir

1. **Penerusan `tool_results**`: Struktur blok data balasan diatur di dalam sub-array berformat khusus yang mengikat ID permintaan asli (`tool_use_id`). Hal ini memudahkan Claude mengidentifikasi hasil komputasi mana yang menjadi jawaban dari instruksi kalkulasi matematika sebelumnya.
2. **Transformasi Jawaban Alami**: Setelah menerima umpan balik fungsi `15 + 27 = 42`, Claude tidak lagi memicu interupsi `stop_reason="tool_use"`. Model akan langsung beralih memproses konten teks penutup yang ramah dan alami (misalnya: *"Hasil dari 15 + 27 adalah 42."*).

## Visualizing Complete Conversation History Flow

Fantastic work building the complete tool execution pipeline! Now it's time to peek behind the curtain and see exactly how the entire conversation flows from start to finish by implementing a comprehensive message history display system.

Your mission is to create a detailed visualization that shows every step of the tool execution cycle. The conversation history contains fascinating details about how Claude's requests, your tool executions, and the final responses all connect together.

Here's what you need to implement:

    Create a for loop using enumerate() to iterate through each message in the messages array.
    For each message, print the message number and role in a clear format.
    Use isinstance() to check whether the message content is a string or a list.
    If the content is a string (like user messages), print it directly.
    If the content is a list (like tool results or mixed content), create a nested loop with enumerate() to print each content item with proper numbering.

This visualization will give you incredible insight into how Claude agents maintain context and transform raw function outputs into intelligent conversations!

```
import json
from anthropic import Anthropic
from functions import sum_numbers, multiply_numbers

# Initialize the Anthropic client
client = Anthropic()

# Choose a Claude model
model = "claude-sonnet-4-6"

# System prompt with explicit instructions to use tools
system_prompt = (
    "You are a helpful math assistant. "
    "Always use the available tools to perform calculations accurately."
)

# Load the schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a dictionary mapping tool names to functions
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers
}

# Create a list of messages to send to Claude
messages = [
    {"role": "user", "content": "Please calculate 15 + 27"}
]

# Send the messages to Claude
response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    system=system_prompt,
    tools=tool_schemas
)

# Add the assistant's response to messages
messages.append({
    "role": "assistant", 
    "content": response.content
})

# Check if Claude wants to use any functions and execute them
if response.stop_reason == "tool_use":
    # Define a list to collect all tool results
    tool_results = []

    for content_item in response.content:
        # Check if the content item is a tool use
        if content_item.type == "tool_use":
            # Get the tool name, input, and id
            tool_name = content_item.name
            tool_input = content_item.input
            tool_id = content_item.id
            
            # Print the tool name, input
            print(f"Executing: {tool_name}({tool_input})")
            
            # Use try-except to handle both missing tools and execution errors
            try:
                # Check if the requested tool exists and execute it
                if tool_name in tools:
                    result = tools[tool_name](**tool_input)
                else:
                    result = f"Error: Function {tool_name} not found"
            except Exception as e:
                # Handle any errors that occur during function execution
                result = f"Error executing {tool_name}: {str(e)}"
     
            # Print the result
            print(f"Result: {result}")

            # Append a properly structured tool_result for this specific tool_use
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tool_id,
                "content": str(result)
            })

    # Add all tool results as a single user message
    messages.append({
        "role": "user",
        "content": tool_results
    })
    
    # Send the results back to Claude for the final response
    final_response = client.messages.create(
        model=model,
        max_tokens=2000,
        messages=messages,
        system=system_prompt,
        tools=tool_schemas
    )

    # Append the final response to messages
    messages.append({
        "role": "assistant",
        "content": final_response.content
    })    

    # Print the final response
    print("\nClaude's final response:")
    print(final_response.content[0].text)

else:
    # Print the response if Claude did not use any tools
    print("Claude did not use any tools:")
    print(response.content[0].text)

# Print the messages history
print("\nMessages history:")

# TODO: Create a for loop with enumerate to iterate through the messages array
    # TODO: Print the message number and role for each message
    
    # TODO: Use isinstance to check if the content is a string or list
        # TODO: If content is a string, print it directly
        
    # TODO: If content is a list, create a nested loop with enumerate
        # TODO: Print each content item with proper numbering
```

To complete your message history visualization system, you need to iterate through the accumulation of messages and cleanly differentiate between plain text strings (like your initial question) and structured block arrays (such as Claude's tool execution calls and your function outcomes).

Here is the complete Python script with the final visualization block implemented at the bottom:

```python
import json
from anthropic import Anthropic
from functions import sum_numbers, multiply_numbers

# Initialize the Anthropic client
client = Anthropic()

# Choose a Claude model
model = "claude-sonnet-4-6"

# System prompt with explicit instructions to use tools
system_prompt = (
    "You are a helpful math assistant. "
    "Always use the available tools to perform calculations accurately."
)

# Load the schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a dictionary mapping tool names to functions
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers
}

# Create a list of messages to send to Claude
messages = [
    {"role": "user", "content": "Please calculate 15 + 27"}
]

# Send the messages to Claude
response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    system=system_prompt,
    tools=tool_schemas
)

# Add the assistant's response to messages
messages.append({
    "role": "assistant", 
    "content": response.content
})

# Check if Claude wants to use any functions and execute them
if response.stop_reason == "tool_use":
    # Define a list to collect all tool results
    tool_results = []

    for content_item in response.content:
        # Check if the content item is a tool use
        if content_item.type == "tool_use":
            # Get the tool name, input, and id
            tool_name = content_item.name
            tool_input = content_item.input
            tool_id = content_item.id
            
            # Print the tool name, input
            print(f"Executing: {tool_name}({tool_input})")
            
            # Use try-except to handle both missing tools and execution errors
            try:
                # Check if the requested tool exists and execute it
                if tool_name in tools:
                    result = tools[tool_name](**tool_input)
                else:
                    result = f"Error: Function {tool_name} not found"
            except Exception as e:
                # Handle any errors that occur during function execution
                result = f"Error executing {tool_name}: {str(e)}"
     
            # Print the result
            print(f"Result: {result}")

            # Append a properly structured tool_result for this specific tool_use
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tool_id,
                "content": str(result)
            })

    # Add all tool results as a single user message
    messages.append({
        "role": "user",
        "content": tool_results
    })
    
    # Send the results back to Claude for the final response
    final_response = client.messages.create(
        model=model,
        max_tokens=2000,
        messages=messages,
        system=system_prompt,
        tools=tool_schemas
    )

    # Append the final response to messages
    messages.append({
        "role": "assistant",
        "content": final_response.content
    })    

    # Print the final response
    print("\nClaude's final response:")
    print(final_response.content[0].text)

else:
    # Print the response if Claude did not use any tools
    print("Claude did not use any tools:")
    print(response.content[0].text)

# Print the messages history
print("\n==========================================")
print("           MESSAGES HISTORY FLOW")
print("==========================================")

# Create a for loop with enumerate to iterate through the messages array
for i, message in enumerate(messages, start=1):
    # Print the message number and role for each message
    print(f"\n[Message #{i}] Role: {message['role'].upper()}")
    content = message['content']
    
    # Use isinstance to check if the content is a string or list
    if isinstance(content, str):
        # If content is a string, print it directly
        print(f"  Text: {content}")
        
    elif isinstance(content, list):
        # If content is a list, create a nested loop with enumerate
        for j, item in enumerate(content, start=1):
            print(f"  Content Block {j}:")
            
            # Safely handle both standard dictionaries and Anthropic Pydantic block objects
            if hasattr(item, "model_dump"):
                item_data = item.model_dump()
            else:
                item_data = item
                
            print(json.dumps(item_data, indent=4).replace("\n", "\n  "))

```

### 🔍 How to Read the Visualization Output

When running this block, notice how the data representation shifts during each turn:

* **Message #1 (User)**: Outputs clean, simple plain text asking for the calculation.
* **Message #2 (Assistant)**: Shows a structured block list of type `"tool_use"`. Claude chooses the specific function parameters here.
* **Message #3 (User)**: Passes a structured list of type `"tool_result"`, injecting the programmatic function return back into the runtime loop.
* **Message #4 (Assistant)**: Displays a plain text block containing Claude's natural, polished final conversational answer using the calculated context.